# Preprocessing — Colab runner

Runs `preprocess.py` on your **generated** data (the `data/` folder you saved to Drive) and produces the leakage-controlled `processed/` outputs that the training repo consumes: `feature_table.parquet`, `windows_*.npz`, `class_weights.json`, `split_manifest.csv`.

Order: clone → install → mount Drive → set `DATA` to your generated-data folder → run → save `processed/` back to Drive. **Edit `REPO_URL` and `DATA`.**

In [ ]:
import os
import getpass

REPO_OWNER = "savvy-sam"
REPO_NAME = "sewer-blockage"

PAT = os.environ.get("GITHUB_PAT") or getpass.getpass("Enter your GitHub PAT: ")
REPO_URL = f"https://{PAT}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

name = REPO_NAME

if os.path.isdir(name):
    !cd {name} && git remote set-url origin {REPO_URL} && git pull --ff-only
else:
    !git clone {REPO_URL}

%cd {name}

In [ ]:
!pip -q install -r requirements.txt scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Run preprocessing

`DATA` must be the folder that contains `runs/*.parquet` and `manifest.csv` (i.e. wherever you saved the generator's output). The script prints the run-level split, per-split class balance, class weights, knob-correlation report and the leakage-audit result.

In [ ]:
DATA = '/content/drive/MyDrive/sewer_blockage_data'   # <-- folder with runs/ + manifest.csv
!python preprocess.py --data '{DATA}' --out processed --window 60 --stride 10

## Inspect the output

In [ ]:
import pandas as pd, json, glob, os
split_manifest = pd.read_csv('processed/split_manifest.csv')
print('run-level split:')
print(split_manifest['split'].value_counts().to_string())
ft = pd.read_parquet('processed/feature_table.parquet')
print('\nfeature_table:', ft.shape)
print('window files:', [os.path.basename(f) for f in sorted(glob.glob('processed/windows_*.npz'))])
print('class weights (train):', json.load(open('processed/class_weights.json')))

In [ ]:
!cp -r processed '/content/drive/MyDrive/processed'
print('saved -> MyDrive/processed  (point the training notebook DATA at this folder)')